# Agent Communication Matrix - Embedding Pipeline

This notebook demonstrates the embedding generation using Ollama (Pattern 1 - MCP Tool-Call Latency).

In [1]:
import os
import sys
import time
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
project_root = Path().cwd().parent
sys.path.insert(0, str(project_root))

## Step 1: Connect to Ollama

In [2]:
import requests

ollama_host = os.getenv('OLLAMA_URL', 'http://localhost:11434')
ollama_model = os.getenv('EMBED_MODEL', 'nomic-embed-text')

print(f"Ollama Host: {ollama_host}")
print(f"Model: {ollama_model}")

try:
    response = requests.get(f"{ollama_host}/api/tags", timeout=5)
    models = response.json().get('models', [])
    print(f"\n✓ Connected to Ollama")
    print(f"Available models: {[m['name'] for m in models]}")
except Exception as e:
    print(f"✗ Could not connect to Ollama: {e}")

Ollama Host: http://localhost:11434
Model: nomic-embed-text

✓ Connected to Ollama
Available models: ['llama3.2:latest', 'nomic-embed-text:latest']


## Step 2: Test Embedding Generation

In [3]:
# Sample text to embed
test_texts = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning models require large amounts of data",
    "Vector embeddings enable semantic search capabilities"
]

embeddings = []
latencies = []

for text in test_texts:
    start = time.time()
    try:
        response = requests.post(
            f"{ollama_host}/api/embed",
            json={"model": ollama_model, "input": text},
            timeout=30
        )
        latency = (time.time() - start) * 1000  # ms
        
        if response.status_code == 200:
            embedding = response.json()['embeddings'][0]
            embeddings.append(embedding)
            latencies.append(latency)
            print(f"✓ {len(embedding)}-dim embedding | {latency:.1f}ms")
        else:
            print(f"✗ Error: {response.status_code}")
    except Exception as e:
        print(f"✗ Failed: {e}")

if latencies:
    print(f"\nLatency Stats (ms):")
    print(f"  Median: {np.median(latencies):.1f}")
    print(f"  Mean: {np.mean(latencies):.1f}")
    print(f"  Min: {np.min(latencies):.1f}")
    print(f"  Max: {np.max(latencies):.1f}")

✓ 768-dim embedding | 436.7ms
✓ 768-dim embedding | 166.1ms
✓ 768-dim embedding | 297.7ms

Latency Stats (ms):
  Median: 297.7
  Mean: 300.2
  Min: 166.1
  Max: 436.7


## Step 3: Embedding Statistics

In [4]:
if embeddings:
    embedding_array = np.array(embeddings)
    
    print(f"Embedding Shape: {embedding_array.shape}")
    print(f"\nEmbedding Statistics:")
    print(f"  Mean: {np.mean(embedding_array):.6f}")
    print(f"  Std Dev: {np.std(embedding_array):.6f}")
    print(f"  Min: {np.min(embedding_array):.6f}")
    print(f"  Max: {np.max(embedding_array):.6f}")
    
    # Compute cosine similarities
    from sklearn.metrics.pairwise import cosine_similarity
    
    similarities = cosine_similarity(embedding_array)
    print(f"\nCosine Similarities:")
    for i in range(len(test_texts)):
        for j in range(i+1, len(test_texts)):
            sim = similarities[i][j]
            print(f"  Text {i+1} vs Text {j+1}: {sim:.4f}")

Embedding Shape: (3, 768)

Embedding Statistics:
  Mean: 0.000205
  Std Dev: 0.036084
  Min: -0.152342
  Max: 0.151794

Cosine Similarities:
  Text 1 vs Text 2: 0.3414
  Text 1 vs Text 3: 0.2961
  Text 2 vs Text 3: 0.4944


## Step 4: Batch Embedding Processing

In [5]:
# Demonstrate batch processing
batch_size = 10
print(f"Processing in batches of {batch_size}...\n")

batch_latencies = []
num_batches = 3

for batch_num in range(num_batches):
    batch_texts = test_texts * (batch_size // len(test_texts) + 1)
    batch_texts = batch_texts[:batch_size]
    
    start = time.time()
    try:
        response = requests.post(
            f"{ollama_host}/api/embed",
            json={"model": ollama_model, "input": batch_texts},
            timeout=60
        )
        batch_latency = (time.time() - start) * 1000
        batch_latencies.append(batch_latency)
        
        if response.status_code == 200:
            num_embeddings = len(response.json()['embeddings'])
            per_item = batch_latency / num_embeddings
            print(f"Batch {batch_num+1}: {batch_latency:.1f}ms total | {per_item:.1f}ms per item")
    except Exception as e:
        print(f"Batch {batch_num+1} failed: {e}")

if batch_latencies:
    print(f"\nBatch Processing Average: {np.mean(batch_latencies):.1f}ms")

Processing in batches of 10...

Batch 1: 2532.6ms total | 253.3ms per item
Batch 2: 2651.1ms total | 265.1ms per item
Batch 3: 2184.1ms total | 218.4ms per item

Batch Processing Average: 2455.9ms
